<a href="https://colab.research.google.com/github/fipsmaker/masteryproject/blob/main/load_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
from google.colab import userdata
from sqlalchemy import create_engine, text

db_url = userdata.get("TRAVELTIDE_DATABASE_URL")
db_url = db_url.replace("postgres://", "postgresql+psycopg2://", 1)

engine = create_engine(db_url, connect_args={"connect_timeout": 10})

with engine.connect() as conn:
    sessions = pd.read_sql_query(text("""
        SELECT user_id, session_id, session_start
        FROM sessions
        LIMIT 5
    """), conn)

display(sessions)

,user_id,session_id,session_start
0,0,0-a84a2a1eaac74191a1d88b6a0b7231c3,2021-04-01 04:57:00
1,1,1-a6b5c66a03f44324987519206329a065,2021-04-01 19:54:00
2,2,2-cbb7f95795f146d49274f1b6cdebd2d9,2021-04-01 07:58:00
3,3,3-9c9d020b3d4549229664f38a2468ae56,2021-04-01 20:03:00
4,4,4-057b447eb36242e29e758b016e28ea12,2021-04-01 00:42:00


In [8]:
top20_sql = """
WITH analysis_period AS (
    SELECT MAX(session_start) - INTERVAL '2 years' AS start_at
    FROM sessions
),
flight_quantities AS (
    SELECT trip_id, SUM(seats) AS seats
    FROM flights
    GROUP BY trip_id
),
hotel_quantities AS (
    SELECT trip_id, SUM(rooms) AS rooms
    FROM hotels
    GROUP BY trip_id
),
valid_sessions AS (
    SELECT
        s.user_id,
        s.session_id,
        s.session_start,
        s.trip_id,
        s.flight_booked,
        s.hotel_booked
    FROM sessions AS s
    WHERE s.user_id IS NOT NULL
      AND s.session_id IS NOT NULL
      AND s.trip_id IS NOT NULL
      AND NOT EXISTS (
          SELECT 1
          FROM sessions AS c
          WHERE c.trip_id = s.trip_id
            AND c.cancellation IS TRUE
      )
),
first_flight_booking AS (
    SELECT DISTINCT ON (user_id, trip_id)
        user_id, session_id, session_start, trip_id
    FROM valid_sessions
    WHERE flight_booked IS TRUE
    ORDER BY user_id, trip_id, session_start NULLS LAST, session_id
),
first_hotel_booking AS (
    SELECT DISTINCT ON (user_id, trip_id)
        user_id, session_id, session_start, trip_id
    FROM valid_sessions
    WHERE hotel_booked IS TRUE
    ORDER BY user_id, trip_id, session_start NULLS LAST, session_id
),
booking_quantities AS (
    SELECT b.user_id, b.session_id, f.seats, 0 AS rooms
    FROM first_flight_booking AS b
    JOIN flight_quantities AS f USING (trip_id)
    CROSS JOIN analysis_period AS p
    WHERE b.session_start >= p.start_at

    UNION ALL

    SELECT b.user_id, b.session_id, 0 AS seats, h.rooms
    FROM first_hotel_booking AS b
    JOIN hotel_quantities AS h USING (trip_id)
    CROSS JOIN analysis_period AS p
    WHERE b.session_start >= p.start_at
),
per_session AS (
    SELECT
        user_id,
        session_id,
        COALESCE(SUM(seats), 0) AS seats,
        COALESCE(SUM(rooms), 0) AS rooms
    FROM booking_quantities
    GROUP BY user_id, session_id
)
SELECT
    user_id,
    SUM(seats) AS total_seats,
    SUM(rooms) AS total_rooms,
    SUM(seats + rooms) AS total_seats_and_rooms,
    COUNT(*) FILTER (WHERE seats = 1) AS sessions_one_seat,
    COUNT(*) FILTER (WHERE seats >= 2) AS sessions_multiple_seats,
    COUNT(*) FILTER (WHERE rooms = 1) AS sessions_one_room,
    COUNT(*) FILTER (WHERE rooms >= 2) AS sessions_multiple_rooms,
    COUNT(*) FILTER (WHERE seats >= 2 AND rooms >= 2)
        AS sessions_multiple_seats_and_rooms
FROM per_session
GROUP BY user_id
HAVING SUM(seats + rooms) > 0
ORDER BY total_seats_and_rooms DESC, user_id
LIMIT 20;
"""

with engine.connect() as conn:
    top20 = pd.read_sql_query(text(top20_sql), conn)

display(top20)

,user_id,total_seats,total_rooms,total_seats_and_rooms,sessions_one_seat,sessions_multiple_seats,sessions_one_room,sessions_multiple_rooms,sessions_multiple_seats_and_rooms
0,47505,19.0,15.0,34.0,3,6,4,5,5
1,170995,17.0,17.0,34.0,6,4,6,4,4
2,58913,15.0,16.0,31.0,3,5,4,5,5
3,485544,16.0,15.0,31.0,6,3,5,3,3
4,36706,15.0,15.0,30.0,4,3,4,3,3
5,121059,15.0,14.0,29.0,5,3,4,3,3
6,144345,15.0,14.0,29.0,7,4,6,4,4
7,14591,14.0,14.0,28.0,4,3,4,3,3
8,26720,14.0,14.0,28.0,7,3,7,3,3
9,31190,17.0,11.0,28.0,3,2,3,2,2
